# Load Libraries

In [1]:
import os
import warnings
import logging
import sys
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import dotenv
import pyet
import matplotlib.pyplot as plt


# Load environment variables from .env file
dotenv.load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO)

# Suppress warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)

# Load Data

In [2]:
# Load Parquet Data
data = pd.read_parquet('../../data/Iran_Monthly_ETo_1951_2025.parquet')
logging.info("Data loaded from parquet file.")    

INFO:root:Data loaded from parquet file.


In [3]:
data

,year,month,region_id,region_name,station_id,station_name,lat,lon,station_elevation,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,date,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,1951,1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-01-01,NaN,NaN,NaN,NaN
1,1951,2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-02-01,NaN,NaN,NaN,NaN
2,1951,3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-03-01,NaN,NaN,NaN,NaN
3,1951,4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-04-01,NaN,NaN,NaN,NaN
4,1951,5,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-05-01,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,2025,5,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,2025-05-01,NaN,147.74,110.63,121.41
694274,2025,6,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,2025-06-01,NaN,207.46,125.91,138.41
694275,2025,7,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,2025-07-01,NaN,232.24,149.49,166.81
694276,2025,8,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,2025-08-01,NaN,213.22,145.58,162.03


# Data Description

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694278 entries, 0 to 694277
Data columns (total 32 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   year               694278 non-null  int32         
 1   month              694278 non-null  int32         
 2   region_id          694278 non-null  object        
 3   region_name        694278 non-null  object        
 4   station_id         694278 non-null  object        
 5   station_name       694278 non-null  object        
 6   lat                694278 non-null  float64       
 7   lon                694278 non-null  float64       
 8   station_elevation  694278 non-null  float64       
 9   tmax               162557 non-null  float64       
 10  tmax_count         694278 non-null  int64         
 11  tmin               161808 non-null  float64       
 12  tmin_count         694278 non-null  int64         
 13  tm                 161595 non-null  float64 

In [5]:
data = data[[
    'region_id', 'region_name', 'station_id', 'station_name',
    'lat', 'lon', 'station_elevation',
    'date', 'year', 'month',
    'tmax', 'tmax_count', 'tmin', 'tmin_count', 'tm', 'tm_count', 
    'umax', 'umax_count', 'umin', 'umin_count', 'um', 'um_count',
    'ffm', 'ffm_count', 'sshn', 'sshn_count', 'rrr24', 'rrr24_count',
    'FAO56', 'Hargreaves', 'Blaney_Criddle', 'Oudin'
]]

data

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-03-01,1951,3,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-04-01,1951,4,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-05-01,1951,5,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-05-01,2025,5,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,NaN,147.74,110.63,121.41
694274,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-06-01,2025,6,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,NaN,207.46,125.91,138.41
694275,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-07-01,2025,7,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,NaN,232.24,149.49,166.81
694276,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-08-01,2025,8,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,NaN,213.22,145.58,162.03


In [6]:
print(f"Number of unique regions: {data.region_name.nunique()}")
print(f"Region names: {data.region_name.unique().tolist()}")

Number of unique regions: 32
Region names: ['Airforce', 'Alborz', 'Ardebil', 'Azarbayjan-E-Gharbi', 'Azarbayjan-E-Sharghi', 'Bushehr', 'Chaharmahal Va Bakhtiari', 'Esfahan', 'Fars', 'Gilan', 'Golestan', 'Hamedan', 'Hormozgan', 'Ilam', 'Kerman', 'Kermanshah', 'Khohgiluyeh Va Boyerahmad', 'Khorasan Razavi', 'Khuzestan', 'Kordestan', 'Lorestan', 'Markazi', 'Mazandaran', 'North Khorasan', 'Qazvin', 'Qom', 'Semnan', 'Sistan Va Baluchestan', 'South Khorasan', 'Tehran', 'Yazd', 'Zanjan']


In [7]:
d = data[['region_id', 'region_name', 'station_name', 'station_id', 'lat', 'lon', 'station_elevation']].drop_duplicates().reset_index(drop=True)
d[d.duplicated(subset=['region_name', 'station_name'], keep=False)]

,region_id,region_name,station_name,station_id,lat,lon,station_elevation
359,OIKK,Kerman,Golbaf,19680,29.85,57.73,1665.00
360,OIKK,Kerman,Golbaf,99583,29.85,57.73,1665.00
519,OICK,Lorestan,Rymaleh,19124,33.64,48.40,1650.00
520,OICK,Lorestan,Rymaleh,99471,33.64,48.40,1650.00
556,MASA,Mazandaran,Nowshahr,24002,36.95,51.66,9999.00
557,MASA,Mazandaran,Nowshahr,40734,36.66,51.47,-20.90
583,OIMN,North Khorasan,Shirvan,18203,37.38,57.92,1187.00
584,OIMN,North Khorasan,Shirvan,99270,37.43,57.83,1051.00
631,OIIS,Semnan,Shahmirzad,90250,35.78,53.37,1960.00
632,OIIS,Semnan,Shahmirzad,99386,35.77,53.35,1969.00


In [30]:
print(f"Number of unique stations:")
data[['region_name', 'station_name']].drop_duplicates().reset_index(drop=True)

Number of unique stations:


,region_name,station_name
0,Airforce,Dowshan Tappeh
1,Airforce,Hamedan (Nozheh)
2,Airforce,Khurbirjand
3,Airforce,Konarak (Airport)
4,Alborz,Asara
...,...,...
763,Zanjan,Soltaniyeh
764,Zanjan,Zanjan
765,Zanjan,Zanjan (Airport)
766,Zanjan,Zarinabad(Egrood)


In [31]:
# change *_count of days to percent base on number of days in month, year
def days_in_month(year, month):
    if month == 2:
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            return 29
        else:
            return 28
    elif month in [4, 6, 9, 11]:
        return 30
    else:
        return 31

data['days_in_month'] = data.apply(lambda row: days_in_month(row['year'], row['month']), axis=1)

for col in data.columns:
    if col.endswith('_count'):
        percent_col = col.replace('_count', '_percent')
        data[percent_col] = round((data[col] / data['days_in_month']) * 100, 1)


In [32]:
data

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin,days_in_month,tmax_percent,tmin_percent,tm_percent,umax_percent,umin_percent,um_percent,ffm_percent,sshn_percent,rrr24_percent
0,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,28,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-03-01,1951,3,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-04-01,1951,4,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,30,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-05-01,1951,5,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-05-01,2025,5,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,NaN,147.74,110.63,121.41,31,41.90,32.30,32.30,0.00,0.00,0.00,0.00,0.00,35.50
694274,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-06-01,2025,6,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,NaN,207.46,125.91,138.41,30,100.00,93.30,93.30,0.00,0.00,0.00,0.00,0.00,96.70
694275,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-07-01,2025,7,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,NaN,232.24,149.49,166.81,31,100.00,96.80,96.80,0.00,0.00,0.00,0.00,0.00,96.80
694276,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-08-01,2025,8,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,NaN,213.22,145.58,162.03,31,64.50,35.50,35.50,0.00,0.00,0.00,0.00,0.00,61.30


In [33]:
data.dropna(subset=['Hargreaves'])

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin,days_in_month,tmax_percent,tmin_percent,tm_percent,umax_percent,umin_percent,um_percent,ffm_percent,sshn_percent,rrr24_percent
255,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-04-01,1972,4,23.03,30,11.13,30,17.08,30,51.03,30,23.53,30,33.47,30,2.71,30,8.49,30,22.64,30,146.40,121.17,88.96,96.67,30,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
256,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-05-01,1972,5,24.03,31,12.45,31,18.24,31,60.23,31,27.42,31,40.74,31,3.26,31,8.36,31,56.24,31,163.74,142.52,107.30,117.42,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
257,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-06-01,1972,6,32.87,30,20.43,30,26.65,30,37.07,30,14.97,30,23.41,30,3.18,30,11.04,30,8.42,30,244.44,184.87,145.48,162.27,30,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
258,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-07-01,1972,7,36.13,31,23.16,31,29.65,31,33.45,31,17.55,31,23.49,31,2.35,31,11.79,31,2.00,31,245.63,204.14,160.19,179.97,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
259,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-08-01,1972,8,32.61,31,20.26,31,26.44,31,38.16,31,16.81,31,24.49,31,1.83,31,10.16,31,14.00,31,191.03,169.11,134.33,148.69,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
694273,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-05-01,2025,5,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,NaN,147.74,110.63,121.41,31,41.90,32.30,32.30,0.00,0.00,0.00,0.00,0.00,35.50
694274,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-06-01,2025,6,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,NaN,207.46,125.91,138.41,30,100.00,93.30,93.30,0.00,0.00,0.00,0.00,0.00,96.70
694275,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-07-01,2025,7,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,NaN,232.24,149.49,166.81,31,100.00,96.80,96.80,0.00,0.00,0.00,0.00,0.00,96.80
694276,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-08-01,2025,8,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,NaN,213.22,145.58,162.03,31,64.50,35.50,35.50,0.00,0.00,0.00,0.00,0.00,61.30


In [34]:
# filter row with *_percent greater than 80% except rrr24_percent
for col in data.columns:
    if col.endswith('_percent') and col != 'rrr24_percent':
        df = data[data[col] >= 75]
        
df

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin,days_in_month,tmax_percent,tmin_percent,tm_percent,umax_percent,umin_percent,um_percent,ffm_percent,sshn_percent,rrr24_percent
255,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-04-01,1972,4,23.03,30,11.13,30,17.08,30,51.03,30,23.53,30,33.47,30,2.71,30,8.49,30,22.64,30,146.40,121.17,88.96,96.67,30,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
256,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-05-01,1972,5,24.03,31,12.45,31,18.24,31,60.23,31,27.42,31,40.74,31,3.26,31,8.36,31,56.24,31,163.74,142.52,107.30,117.42,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
257,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-06-01,1972,6,32.87,30,20.43,30,26.65,30,37.07,30,14.97,30,23.41,30,3.18,30,11.04,30,8.42,30,244.44,184.87,145.48,162.27,30,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
258,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-07-01,1972,7,36.13,31,23.16,31,29.65,31,33.45,31,17.55,31,23.49,31,2.35,31,11.79,31,2.00,31,245.63,204.14,160.19,179.97,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
259,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1972-08-01,1972,8,32.61,31,20.26,31,26.44,31,38.16,31,16.81,31,24.49,31,1.83,31,10.16,31,14.00,31,191.03,169.11,134.33,148.69,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692477,OITZ,Zanjan,88118,Zanjan (Airport),36.77,48.37,1640.70,2025-03-01,2025,3,13.41,31,1.37,31,7.38,31,68.58,31,32.39,31,46.52,31,3.17,31,7.28,30,53.31,31,83.09,71.97,43.40,44.34,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,96.80,100.00
692478,OITZ,Zanjan,88118,Zanjan (Airport),36.77,48.37,1640.70,2025-04-01,2025,4,19.15,30,6.83,30,13.00,30,62.27,30,29.13,30,42.24,30,4.59,30,7.07,29,39.02,30,129.15,107.15,73.11,77.57,30,100.00,100.00,100.00,100.00,100.00,100.00,100.00,96.70,100.00
692479,OITZ,Zanjan,88118,Zanjan (Airport),36.77,48.37,1640.70,2025-05-01,2025,5,26.40,31,9.95,31,18.17,31,55.81,31,19.10,31,32.27,31,3.84,31,10.14,31,30.02,31,188.28,168.92,107.58,116.64,31,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
692481,OITZ,Zanjan,88118,Zanjan (Airport),36.77,48.37,1640.70,2025-07-01,2025,7,32.78,29,18.92,29,25.85,29,46.83,29,21.48,29,30.41,29,5.61,29,10.73,28,8.60,28,278.62,193.82,144.99,159.98,31,93.50,93.50,93.50,93.50,93.50,93.50,93.50,90.30,90.30


In [41]:
# df.query("region_name == ['Khorasan Razavi', 'South Khorasan', 'North Khorasan']").groupby("station_name")["station_name"].count().sort_values(ascending=False).div(12).reset_index(name="count_years")
df.groupby(["region_name", "station_name", "station_id"])["station_name"].count().sort_values(ascending=False).div(12).reset_index(name="count_years")

,region_name,station_name,station_id,count_years
0,Tehran,Tehran (Mehrabad Airport),40754,70.50
1,Khorasan Razavi,Mashhad,40745,65.25
2,Fars,Shiraz,40848,62.17
3,Gilan,Bandar-E-Anzali,40718,61.42
4,Mazandaran,Babolsar,40736,61.08
...,...,...,...,...
384,Tehran,Test,99999,0.25
385,North Khorasan,Raz,99245,0.17
386,Khorasan Razavi,Kalat-E-Nader,99289,0.08
387,Lorestan,Shulabad,99502,0.08


In [22]:
data.query("station_name == 'Golbaf' and region_name == 'Kerman'")

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin,days_in_month,tmax_percent,tmin_percent,tm_percent,umax_percent,umin_percent,um_percent,ffm_percent,sshn_percent,rrr24_percent
322023,OIKK,Kerman,19680,Golbaf,29.85,57.73,1665.00,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
322024,OIKK,Kerman,99583,Golbaf,29.85,57.73,1665.00,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
322025,OIKK,Kerman,19680,Golbaf,29.85,57.73,1665.00,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,28,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
322026,OIKK,Kerman,99583,Golbaf,29.85,57.73,1665.00,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,28,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
322027,OIKK,Kerman,19680,Golbaf,29.85,57.73,1665.00,1951-03-01,1951,3,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
323812,OIKK,Kerman,99583,Golbaf,29.85,57.73,1665.00,2025-07-01,2025,7,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
323813,OIKK,Kerman,19680,Golbaf,29.85,57.73,1665.00,2025-08-01,2025,8,33.13,31,19.18,31,26.15,31,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,22,NaN,182.39,129.87,150.48,31,100.00,100.00,100.00,0.00,0.00,0.00,0.00,0.00,71.00
323814,OIKK,Kerman,99583,Golbaf,29.85,57.73,1665.00,2025-08-01,2025,8,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN,31,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
323815,OIKK,Kerman,19680,Golbaf,29.85,57.73,1665.00,2025-09-01,2025,9,32.46,21,17.94,21,25.20,21,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,20,NaN,154.53,110.23,123.83,30,70.00,70.00,70.00,0.00,0.00,0.00,0.00,0.00,66.70
